# 11 · Risk models: Cox, boosted Cox and DeepSurv (P5)

Graft failure and low ECD after the 12-month visit. Same repeated CV for every model, a simpler model wins near-ties, test once; then calibration and isotonic recalibration (ranking unchanged). Code: `risk/cox.py`, `risk/models.py`, `risk/experiment.py`. Reads `outputs/risk/risk_results.json` (`python -m recommendation.main risk` retrains).

> **Data source:** simulated post-transplant cohort calibrated to published studies (method development), because NEBSL follow-up records are not yet available. The same code runs on real follow-up data (`data_source = "nebsl"`).
>
> **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, Markdown, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
P = cfg["paths"]
OUTPUTS = Path(P["outputs"])
MODELS = Path(P["models"])
print("Cohort:", Path(P["followup_cohort"]).name, "| landmark dataset:", Path(P["followup_dataset"]).name)

In [ ]:
r = json.loads((OUTPUTS / "risk" / "risk_results.json").read_text())
print("Settings (chosen on training grafts):", r["settings"])

## 1. Comparison by repeated cross-validation

In [ ]:
for endpoint, e in r["endpoints"].items():
    print(endpoint, "-> chosen:", e["chosen"])
    display(pd.DataFrame(e["cv"]).T[["c_index", "auc_12m", "auc_24m", "auc_36m", "brier_24m"]].round(3))

## 2. Test grafts (used once)

In [ ]:
for endpoint, e in r["endpoints"].items():
    print(endpoint)
    display(pd.DataFrame(e["test"]).T[["events", "c_index", "auc_24m", "brier_24m"]].round(3))

## 3. Calibration at 24 months: before and after recalibration

In [ ]:
display(Image(OUTPUTS / "risk" / "calibration_24m.png"))
for endpoint, e in r["endpoints"].items():
    rc = e["recalibration"]
    print(f'{endpoint}: calibration slope {rc["slope"]:.2f}; mean gap '
          + ", ".join(f'{w} {rc[w]["gap_before"]:.3f} -> {rc[w]["gap_after"]:.3f}' for w in ("test", "external_simulated_cohort")))

Rejection is constrained to never lower the risk (clinical sign constraint), because only a handful of grafts have a rejection by 12 months.